# 🦓 Practical 2: Image Classification using Convolutional Neural Networks (CNNs) 🦌

**Authors**: Clinton Mwangi, Victor Ruto.

## Introduction

Image classification is one of the most important tasks in computer vision. The goal is simple: given an image, can a computer correctly identify what it contains?

In this practical, we will train a **Convolutional Neural Network (CNN)** to recognise different species of animals from camera trap images collected in the wild. By the end of the session, we will have built, trained and evaluated our own deep learning model using **PyTorch**.

Although modern computer vision models can contain hundreds of layers and millions of parameters, the fundamental training pipeline remains largely the same. In this practical, we will focus on understanding this workflow using a small CNN inspired by the VGG architecture.

Rather than treating deep learning as a "black box", we will build each component ourselves and observe how the model learns from data.

---



## 🎯 Learning Objectives

By the end of this practical you should be able to:

- Load an image dataset using PyTorch.
- Prepare images for training using transforms and data loaders.
- Build a simple Convolutional Neural Network.
- Train a CNN for image classification.
- Monitor training using loss and accuracy curves.
- Evaluate the performance of the trained model.
- Visualise predictions made by the model.
- Use the model to classify unseen camera trap images.
---

## Workflow

During this practical we will follow the typical workflow used in many deep learning projects.

```
Dataset  
    ▼  
Explore Images   
    ▼  
Preprocess Images   
    ▼  
Build CNN  
    ▼  
Train Model  
    ▼  
Evaluate Model  
    ▼  
Visualise Predictions

```

By completing these steps, you will gain practical experience with the complete image classification pipeline used in modern computer vision applications.

## 📦 1. Importing the Required Libraries

Before we can build and train a neural network, we first need to import the Python libraries that provide the necessary functionality.

In this practical, we will primarily use **PyTorch**, one of the most widely used deep learning frameworks. PyTorch provides tools for building neural networks, loading datasets, training models and performing inference.

We will also use torchvision, a foundational companion library for PyTorch designed for computer vision applications that provides highly optimized transformation utilities for data augmentation

We will also use several supporting libraries for data visualisation and image processing.

In [ ]:
%%capture
!pip install torchinfo torchmetrics

In [ ]:
%%capture
!sudo apt install tree

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

import torchmetrics
from torchinfo import summary

## 📂 2. The Dataset: DSAIL-Porini

DSAIL-Porini is a dataset of animal images collected at the DeKUTWC using low-cost camera traps developed locally at DSAIL.

Camera traps are autonomous motion activated cameras that are deployed in the wild to capture images of animals in their natural habitat with little interruption to their peace. They automatically detect motion and trigger to capture images of animals when they pass in front of them.

<div style="text-align: center;">
    <img src="https://github.com/vickruto/assets/releases/download/v0.0.1/porini-sample-images.png"  style="height:400px; width:auto;">
</div>

**Figure 1:** Species covered in the DSAIL Porini dataset : `(a) waterbuck`, `(b) sykes monkey`, `(c) warthog`, `(d) bushbuck`,`(e) zebra` and `(f) impala`.

For this practical session, we will be training a CNN with a curated subset of the dataset constiting of 400 images sample from the 6 species.

<table>
<tr>
<td align="center">

<img src="https://github.com/vickruto/assets/releases/download/v0.0.1/dsail_camera_trap_deployment.jpg" height="300"/>

**Figure 2(a):** DSAIL Camera Trap Components

</td>
<td align="center">

<img src="https://github.com/vickruto/assets/releases/download/v0.0.1/dsail_camera_trap_components.png" height="300"/>

**Figure 2(b):** DSAIL Camera Trap in Deployment

</td>
</tr>
</table>

The locally made camera trap used to collect the images
Sample DSAIL-Porini images


You can read more about the dataset here: [DSAIL-Porini](https://www.sciencedirect.com/science/article/pii/S2352340922010666)

### ⏳ **2.1. Loading the Dataset**

Our dataset consists of camera trap images organised into folders, where each folder corresponds to a different animal species.

The dataset has already been split into training, validation and test subsets to ensure that we are evaluating our model using unseen sets of images after each training loop and at the end of our training iterations as we will see.  

PyTorch provides the `ImageFolder` class, which automatically assigns a numerical label to each folder and loads the corresponding images.

The dataset is organized in a classical format for orgnanizing image datasets where the subfolders of the root folder consists of the classes.

In [ ]:
## TODO: Add finalized dataset download link
!wget -nc https://github.com/vickruto/assets/releases/download/v0.0.1/m3-dsail-porini-classification.zip

In [ ]:
!unzip -q m3-dsail-porini-classification.zip -d m3-dsail-porini-classification-1

In [ ]:
# Check the orgzanization of the dataset
!tree -d /content/m3-dsail-porini-classification-1

In [ ]:
# Define the dataset root path & split paths
DATASET_DIR = Path("m3-dsail-porini-classification-1")

TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'valid'
TEST_DIR = DATASET_DIR / 'test'

# Check that the dataset has been downloaded
assert DATASET_DIR.exists(), f"Dataset folder not found! Download it and make sure it is in the current dir ({Path.cwd()})"

In [ ]:
# Get the class names from the subfolder names
class_names = sorted(
    [
        folder.name
        for folder in TRAIN_DIR.iterdir()
        if folder.is_dir()
    ]
)

print(f"Number of classes: {len(class_names)}")
print(class_names)

### **🔎 2.2. Exploring the Dataset**

In [ ]:
!tree -d /content/m3-dsail-porini-classification-1

In [ ]:
image_counts = {}

for class_name in class_names:
    class_dir = TRAIN_DIR / class_name

    num_images = len(
        list(class_dir.glob("*.jpg"))
        + list(class_dir.glob("*.jpeg"))
        + list(class_dir.glob("*.png"))
    )

    image_counts[class_name] = num_images

from pprint import pprint
pprint(image_counts)
print(f'Total:', sum(image_counts.values()))


In [ ]:
class_dir

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(image_counts.keys(), image_counts.values())
plt.xticks(rotation=45)
plt.ylabel("Number of Images")
plt.title("Number of Images per Species")
plt.show()

In [ ]:
num_examples = len(class_names)

fig, axes = plt.subplots(1, num_examples, figsize=(4*num_examples, 3))

axes = axes.flatten()

for ax, class_name in zip(axes, class_names[:num_examples]):

    image_files = list((TRAIN_DIR / class_name).glob("*"))

    image_path = random.choice(image_files)

    image = Image.open(image_path)

    ax.imshow(image)

    ax.set_title(class_name)

    ax.axis("off")

plt.tight_layout()
plt.suptitle("Sample Images of each species from the Dataset", y=1.02)
plt.show()

A short Critique of the dataset from our short exploration ....

- As you can see from the short exploration or our dataset above, the dataset is relatively small (~400 images). Perhaps this is a bit too small for our model to learn to differentiaate between different animals species
- Some of the images are really tricky. In some cases the animals are too far away from the camera and also some images are weidly coloured because of an image artifact
- We can right away appreciate the variety of images due to the following:
  - different lighting
  - different poses
  - occlusion
  - different positioning of the subjects in the image frame
  - partial animals
  - different backgrounds,  
  and many others

These are all factors that can influence the performance of a deep learning model.

## 🔄 3. Preparing the Dataset for Training

The images stored on disk cannot be used directly to train a neural network. Before training, we need to prepare them so that they have a consistent format and can be efficiently loaded into memory.

In this section, we will build the data pipeline that prepares our dataset for training.

The pipeline consists of four main steps:

1. Resize every image to the same dimensions.
2. Convert the images into PyTorch tensors.
3. Load / Split the dataset into training, validation and test sets.
4. Create DataLoaders to efficiently load images during training.

###  🖼️ 3.1. Image Transformations

Camera trap images, just like any other images do not come in the same size. Some images may be 1920 × 1080 pixels, while others could be much larger or smaller.

Neural networks, however, expect every image in a batch to have exactly the same dimensions.

We therefore apply a series of **transformations** that prepare each image before it is passed to the network.

In this practical we will use two transformations:

- **Resize** – ensures that every image has the same dimensions.
- **ToTensor** – Normalizes and converts an image into a PyTorch tensor that can be processed by the neural network.

Later in the course, we will introduce additional transformations such as random flipping, rotation and colour augmentation to improve the model's ability to generalise.  

In our case, we will be resizing all images to 224 × 224

```
### Why 224 × 224?

Many popular convolutional neural networks, including VGG, ResNet and EfficientNet, are commonly trained using images resized to **224 × 224 pixels**.

Using a fixed image size simplifies the design of the neural network and allows multiple images to be processed together in a mini-batch.
```

### ✂️ 3.2. Dataset Splits

A machine learning model should not be evaluated using the same images that it was trained on.

Instead, we divide the dataset into three separate subsets.

```
                Entire Dataset
                      │
      ┌───────────────┼───────────────┐
      ▼               ▼               ▼
 Training Set   Validation Set    Test Set
```

Each subset has a different purpose.

| Dataset | Purpose |
|----------|---------|
| **Training** | Used to learn the model parameters. |
| **Validation** | Used to monitor performance during training and choose the best model. |
| **Test** | Used once at the end to estimate how well the model generalises to unseen data. |

In this practical we will use the following split:
- **70%** Training
- **15%** Validation
- **15%** Test
**bold text**
The dataset has already been split, all we need to do is load the splits

In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
train_dataset = ImageFolder(
    root= DATASET_DIR / 'train',
    transform=train_transform,
)


val_dataset = ImageFolder(
    root= DATASET_DIR / 'valid',
    transform=train_transform,
)


test_dataset = ImageFolder(
    root= DATASET_DIR / 'test',
    transform=train_transform,
)

### 🚚 3.3. PyTorch DataLoaders

Training a neural network involves repeatedly loading hundreds to thousands of images from disk.

Rather than loading one image at a time, PyTorch uses a **DataLoader** to automatically group images into **mini-batches**.

Mini-batches make training much more efficient because modern CPUs and GPUs are designed to process many images in parallel.

We will also shuffle the training data before every epoch. This prevents the model from learning the order of the images and generally leads to better training.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
DATASET_DIR = Path() / 'm3-dsail-porini-classification-1'


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

**🔍 Inspecting One Mini-Batch**  

Before we begin building our neural network, let's inspect one mini-batch of images.

Notice that the images are no longer individual files—they have been grouped into a single tensor.

Understanding the shape of this tensor is important because it determines the format expected by our CNN.

In [ ]:
# Retrieve one mini-batch
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
## Plot the images from one min-batch
fig, axes = plt.subplots(4, 8, figsize=(15, 10))
axes = axes.flatten()

for i in range(32):
    image = images[i].permute(1, 2, 0)  # (C, H, W) -> (H, W, C)

    axes[i].imshow(image)
    axes[i].set_title(train_dataset.classes[labels[i]])
    axes[i].axis("off")

plt.tight_layout()
plt.show()

**💡 Understanding the Tensor Shape**

The shape of the image tensor is:

```
(batch_size, channels, height, width)
```

For our dataset this becomes:

```
(32, 3, 224, 224)
```

where:

- **32** is the number of images in the mini-batch.
- **3** represents the Red, Green and Blue (RGB) colour channels.
- **224 × 224** is the size of each image.

This is the format expected by most convolutional neural networks implemented in PyTorch.

## 🧠 4. Building Our First Convolutional Neural Network

Now that our data is ready, we can build the model that will learn to recognise the different animal species.

A **Convolutional Neural Network (CNN)** is a type of neural network designed specifically for analysing images. Unlike a traditional neural network, which treats every pixel independently, a CNN learns to detect useful visual patterns such as edges, textures, shapes and eventually entire objects.

As the image passes through the network, these simple features are gradually combined to form more complex representations. For example, the network may first detect edges, then animal stripes or fur patterns, and finally recognise an entire zebra or impala.

In the next sections, we will build a simplified CNN inspired by the famous **VGG architecture**. Although much smaller than modern networks, it contains all of the key building blocks used in many convolutional neural networks.

### 📐 4.1. TinyVGG Architecture

Our model consists of two convolutional blocks followed by a classifier.

```
Input Image (3 × 224 × 224)

        │
        ▼

┌───────────────────────────────┐
│ Convolution                   │
│ ReLU                          │
│ Convolution                   │
│ ReLU                          │
│ Max Pooling                   │
└───────────────────────────────┘

        │
        ▼

┌───────────────────────────────┐
│ Convolution                   │
│ ReLU                          │
│ Convolution                   │
│ ReLU                          │
│ Max Pooling                   │
└───────────────────────────────┘

        │
        ▼

Adaptive Average Pooling

        │
        ▼

Flatten

        │
        ▼

Fully Connected Layer

        │
        ▼

Predicted Animal Species
```

Although this network is relatively small, it follows the same design principles as many larger convolutional neural networks.

In [ ]:
class TinyVGG(nn.Module):
    """
    A simplified VGG-style Convolutional Neural Network for image classification.

    Architecture:
        Input
            ↓
        Conv → ReLU
            ↓
        Conv → ReLU
            ↓
        MaxPool
            ↓
        Conv → ReLU
            ↓
        Conv → ReLU
            ↓
        MaxPool
            ↓
        Flatten
            ↓
        Linear
            ↓
        Output
    """

    def __init__(
        self,
        input_channels: int = 3,
        hidden_units: int = 32,
        num_classes: int = 8,
    ):
        super().__init__()

        ####################################################################
        # First Convolutional Block
        ####################################################################
        self.conv_block_1 = nn.Sequential(

            # First convolution
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),

            # Second convolution
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),

            # Downsample feature maps
            nn.MaxPool2d(kernel_size=2)
        )

        ####################################################################
        # Second Convolutional Block
        ####################################################################
        self.conv_block_2 = nn.Sequential(

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units * 2,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units * 2,
                out_channels=hidden_units * 2,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)
        )

        ####################################################################
        # Classification Head
        ####################################################################
        #
        # Input image size = 224×224
        #
        # After MaxPool:
        #
        # 224 → 112 → 56
        #
        # Therefore feature maps are:
        #
        # (64, 56, 56)
        #
        ####################################################################
        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1,1)),

            nn.Flatten(),

            nn.Dropout(0.5),

            nn.Linear(
                hidden_units*2,
                num_classes
            )
        )
    def forward(self, x):

        x = self.conv_block_1(x)

        x = self.conv_block_2(x)

        x = self.classifier(x)

        return x



### 🧱 4.2. Architecture Design: Blocks

If you look closely at the TinyVGG architecture, you will notice that it is built by repeating the same pattern of layers.

```
Convolution
     ↓
ReLU
     ↓
Convolution
     ↓
ReLU
     ↓
Max Pooling
```

This raises an interesting question:

> **Why don't we just use one convolutional layer? Why repeat the same block multiple times?**

The answer lies in how neural networks learn to recognise increasingly complex visual patterns.

Each convolutional block extracts features from the output of the previous block. As we move deeper into the network, the features become more abstract and more meaningful.

For example, when classifying camera trap images:

| Network Depth | Example Features Learned |
|----------------|--------------------------|
| Early layers | Edges, corners, simple lines |
| Middle layers | Fur textures, stripes, ears, legs |
| Deeper layers | Animal faces, body shapes, horns |
| Final layer | Complete animal species |

Rather than trying to recognise an entire zebra directly from raw pixels, the network gradually builds its understanding of the image one step at a time.

This idea is known as **hierarchical feature learning**, and it is one of the key reasons why deep neural networks are so successful.

---
Repeating the same building block has several advantages:

- **Simplicity** – Once we know that a particular block works well, we can reuse it multiple times.
- **Progressive Learning** – Each block learns more complex features than the previous one.
- **Scalability** – Larger networks can often be created simply by stacking more blocks.
- **Modularity** – The network becomes easier to design, understand and modify.

### 🚀 4.3. Creating the Model

Now that we have defined the architecture, we can create an instance of our CNN.

We also move the model to an appropriate computation device. If a **Graphics Processing Unit (GPU)** is available, all computations will be performed there, resulting in much faster training. Deep learning models perform a large number of mathematical operations during training. These computations can be accelerated using a GPU, which is specifically designed for highly parallel numerical calculations.

PyTorch allows us to write code that runs on either a CPU or a GPU with only a few small changes. In the following cell, we will automatically select a GPU if one is available; otherwise, the code will run on the CPU.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
NUM_CLASSES = len(class_names)

model = TinyVGG(
    input_channels=3,
    hidden_units=32,
    num_classes=NUM_CLASSES,
)

model = model.to(device)
print(model)

### **📊 4.4. Inspecting the Model**

Before training the model, it is useful to inspect its architecture.

The summary below shows how the dimensions of the data change as it passes through each layer, as well as the number of trainable parameters in the network.

In [ ]:
from torchinfo import summary

summary(
    model,
    input_size=(1, 3, 224, 224),
    col_names=[
        "input_size",
        "output_size",
        "num_params"
    ],
)

### **✅ 4.5. Testing the Forward Pass**

Before we begin training, it is good practice to verify that the model can process an image and produce an output of the expected shape.

Each value in the output corresponds to one animal species. These values are known as **logits** and represent the model's confidence before they are converted into probabilities.

In [ ]:
# Create a dummy input image
dummy_image = torch.randn(1, 3, 224, 224).to(device)

# Pass the image through the network
output = model(dummy_image)

print(f"Output shape: {output.shape}")

## 🚀 5. Training the Convolutional Neural Network

Our CNN has now been built, but it is not yet capable of recognising animals.

At this stage, all of the weights inside the network have been **randomly initialised**. As a result, the model's predictions are essentially random.

The goal of training is to gradually adjust these weights so that the model makes increasingly accurate predictions.

To achieve this, every neural network requires three key components:

1. **A Loss Function** – measures how far the predictions are from the correct answers.
2. **An Optimizer** – updates the model's weights to reduce the loss.
3. **A Training Loop** – repeatedly presents batches of images to the model and allows it to learn from its mistakes.

Together, these components enable the CNN to improve its performance over many training iterations.


### **🎯 5.1. Choosing a Loss Function**

When the CNN processes an image, it produces a prediction.

To improve the model, we first need a way of measuring **how good or bad that prediction is**.

This is the role of the **loss function**.

A loss function compares the model's prediction with the correct label and produces a single number called the **loss**.

- A **small loss** indicates that the prediction was close to the correct answer.
- A **large loss** indicates that the prediction was poor.

During training, the model tries to minimise this loss.

For image classification problems where each image belongs to exactly one class, the most commonly used loss function is **Cross-Entropy Loss**.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

**Why Cross-Entropy Loss?**

Our task is a **multi-class classification problem**, where each image belongs to exactly one of several animal species.

Cross-Entropy Loss compares the predicted class probabilities with the correct class label and assigns a larger penalty to incorrect predictions.

It is one of the most widely used loss functions in deep learning for image classification.

### ⚡ 5.2. Choosing an Optimizer

Once the loss has been calculated, the model needs to decide **how to improve**.

This is the role of the **optimizer**.

The optimizer examines the gradients computed during backpropagation and updates the model's weights in the direction that reduces the loss.

You can think of the optimizer as the learning algorithm that teaches the network how to improve after every mini-batch of images.

In this practical, we will use the **Adam optimizer**, which is one of the most popular optimization algorithms in deep learning because it is fast, reliable and generally works well for many problems.

In [ ]:
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,

    # Small amount of regularisation to reduce overfitting.
    # weight_decay=1e-4,
)

**Understanding the Learning Rate**

The **learning rate** determines how large a step the optimizer takes when updating the model's weights.

Choosing an appropriate learning rate is important:

- A learning rate that is **too large** may cause training to become unstable.
- A learning rate that is **too small** may result in very slow learning.

Although the best value depends on the problem, **0.001** is a common starting point when using the Adam optimizer.

## 🔁 6. Implementing the Training Loop

In the previous section, we introduced the three components required to train a neural network:

- A loss function
- An optimizer
- A training loop

We will now combine these components into a complete training pipeline.

Rather than writing all the code in a single block, we will divide the training process into smaller functions. This makes the code easier to understand, debug and reuse.

The training pipeline consists of three stages:

1. Train the model for one epoch.
2. Evaluate the model on the validation dataset.
3. Repeat this process for multiple epochs while monitoring performance.

---

Training a neural network is a repetitive process.

For every mini-batch of images, the following steps are performed:

```
Mini-batch of Images
          │
          ▼
Forward Pass
(Model makes predictions)
          │
          ▼
Compute Loss
(Compare predictions with labels)
          │
          ▼
Backpropagation
(Compute gradients)
          │
          ▼
Optimizer Step
(Update the weights)
          │
          ▼
Repeat...
```

Each complete pass through the entire training dataset is called an **epoch**.

With every epoch, we hope that the model becomes slightly better at recognising animal species.

---

**🧠 Forward Pass vs Backward Pass**

Training a neural network consists of two main phases.

***Forward Pass***

During the forward pass:

- Images are passed through the CNN.
- The CNN produces predictions.
- The loss is calculated by comparing the predictions with the true labels.

***Backward Pass***

During the backward pass:

- PyTorch computes how much each weight contributed to the error.
- Gradients are calculated automatically using **backpropagation**.
- The optimizer updates the weights to reduce the loss.

This forward-and-backward cycle is repeated thousands of times until the model has learned useful visual features.

### **🏋️ 6.1. Training for One Epoch**

An **epoch** is one complete pass through the training dataset.

During an epoch, the model processes every mini-batch of images exactly once.

For each mini-batch, the following steps are performed:

1. Move the images and labels to the selected device.
2. Perform a forward pass to obtain predictions.
3. Compute the loss.
4. Calculate gradients using backpropagation.
5. Update the model weights.
6. Record the training loss and accuracy.

After every mini-batch has been processed, we calculate the average training loss and accuracy for the epoch.

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """
    Train the model for one epoch.

    Returns
    -------
    average_loss : float
        Mean training loss over the epoch.

    average_accuracy : float
        Classification accuracy over the epoch.
    """

    # Put the model into training mode.
    model.train()

    running_loss = 0.0

    accuracy = MulticlassAccuracy(
        num_classes=NUM_CLASSES
    ).to(device)

    for images, labels in dataloader:

        # Move data to the selected device
        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = loss_fn(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track metrics
        running_loss += loss.item() * images.size(0)

        accuracy.update(outputs, labels)

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_accuracy = accuracy.compute().item()

    return epoch_loss, epoch_accuracy

### **📋 6.2. Evaluating the Model**

After each training epoch, we evaluate the model using the validation dataset.

Unlike the training set, the validation set is **never used to update the model weights**.

Instead, it provides an estimate of how well the model performs on unseen images.

This helps us detect problems such as **overfitting**, where the model memorises the training data but performs poorly on new examples.

In [ ]:
@torch.inference_mode()
def validate(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float]:

    model.eval()

    running_loss = 0.0

    accuracy = MulticlassAccuracy(
        num_classes=NUM_CLASSES
    ).to(device)

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = loss_fn(outputs, labels)

        running_loss += loss.item() * images.size(0)

        accuracy.update(outputs, labels)

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_accuracy = accuracy.compute().item()

    return epoch_loss, epoch_accuracy

Note that before our definition of the evaluation function, we have a decorateor `@torch.inference_mode()`

```python
@torch.inference_mode()
def validate(
    model: nn.Module,
    ---
```    

**Why Use `@torch.inference_mode()`?**

During evaluation, we only want the model to make predictions.

Unlike training, we are **not updating the model weights**, so there is no need to compute gradients or keep track of intermediate values required for backpropagation.

Applying the `@torch.inference_mode()` decorator tells PyTorch that the function will only be used for inference.

This provides two important benefits:

- **Reduced memory usage** because gradients do not need to be stored.
- **Faster execution** since PyTorch can skip several computations that are only required during training.

For this reason, `@torch.inference_mode()` is the recommended approach for functions that perform model evaluation or prediction.

> **Note**
>
> `torch.inference_mode()` is similar to `torch.no_grad()`, but it performs additional optimisations that make inference even more efficient. For most evaluation and prediction tasks, it is the preferred choice.

### **💾 6.3. Saving the Best Model**

Training a model for many epochs does not always improve its performance.

Sometimes the validation accuracy reaches its highest value before the final epoch.

For this reason, it is common practice to save the model whenever it achieves a new best validation accuracy.

This process is known as **checkpointing**.

If training is interrupted or later epochs perform worse, we can still recover the best-performing model.

Here we will only be saving the best model but we can choose to save the model after every epoch or after every few epochs.

In [ ]:
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

### **🔄 6.4. The Training Loop**

We are now ready to train the model.

For each epoch we will:

1. Train the model using the training dataset.
2. Evaluate it using the validation dataset.
3. Record the training and validation metrics.
4. Save the model if its validation accuracy improves.

As training progresses, we hope to observe:

- 📉 The training loss decreasing.
- 📈 The training accuracy increasing.
- 📉 The validation loss decreasing.
- 📈 The validation accuracy increasing.

However, these trends may not continue indefinitely. If the training accuracy continues to improve while the validation accuracy stagnates or decreases, the model may be **overfitting** the training data.

In the next section, we will visualise these metrics using learning curves to better understand the model's behaviour.

In [ ]:
# =============================================================================
# Train the TinyVGG model
# =============================================================================

NUM_EPOCHS = 120

# Store training history for plotting later
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

# Track the best model
best_val_acc = 0.0

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):

    ###########################################################################
    # Training
    ###########################################################################

    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
    )

    ###########################################################################
    # Validation
    ###########################################################################

    val_loss, val_acc = validate(
        model=model,
        dataloader=val_loader,
        loss_fn=loss_fn,
        device=device,
    )

    ###########################################################################
    # Save metrics
    ###########################################################################

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    ###########################################################################
    # Save the best model
    ###########################################################################

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        checkpoint_path = CHECKPOINT_DIR / "tinyvgg_best_model.pth"

        torch.save(model.state_dict(), checkpoint_path)

        checkpoint_message = "✓ Saved Best Model"

    else:

        checkpoint_message = ""

    ###########################################################################
    # Display progress
    ###########################################################################

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.3f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.3f} "
        f"{checkpoint_message}"
    )

print("\nTraining complete!")
print(f"Best validation accuracy: {best_val_acc:.3f}")

In [ ]:
## TODO: Add a seconds/timer counter to show the time taken for the training. Perhaps at the most extreme left

## 📈 7. Visualising the Learning Curves

During training, we recorded the model's performance after every epoch.

These values provide useful information about how the model learned over time.

Rather than examining a long list of numbers, we can visualise the training process using **learning curves**.

In this section, we will plot:

- Training loss
- Validation loss
- Training accuracy
- Validation accuracy

These plots help us answer important questions such as:

- Is the model still learning?
- Has the model converged?
- Is the model overfitting the training data?

In [ ]:
def plot_training_history(history: dict) -> None:
    """
    Plot the training and validation loss and accuracy.

    Parameters
    ----------
    history : dict
        Dictionary containing the training history.
    """

    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ---------------------------------------------------------
    # Loss
    # ---------------------------------------------------------
    axes[0].plot(
        epochs,
        history["train_loss"],
        marker="o",
        label="Training"
    )

    axes[0].plot(
        epochs,
        history["val_loss"],
        marker="o",
        label="Validation"
    )

    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross Entropy Loss")
    axes[0].legend()
    axes[0].grid(True)

    # ---------------------------------------------------------
    # Accuracy
    # ---------------------------------------------------------
    axes[1].plot(
        epochs,
        history["train_acc"],
        marker="o",
        label="Training"
    )

    axes[1].plot(
        epochs,
        history["val_acc"],
        marker="o",
        label="Validation"
    )

    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True)

    fig.suptitle("Training and Validation Loss for DSAIL-Porini with Train Transforms RESUMING")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_history(history)

### 📖 7.1. Interpreting the Learning Curves

Learning curves provide valuable insight into how well a model is learning.

### Ideal Training

In an ideal situation:

- Training loss decreases.
- Validation loss also decreases.
- Training and validation accuracy both increase.
- The two curves remain relatively close together.

This indicates that the model is learning useful patterns that generalise well to unseen images.

---

### Overfitting

Sometimes the model performs extremely well on the training set but much worse on the validation set.

Typical signs include:

- Training loss continues to decrease.
- Validation loss begins to increase.
- Training accuracy approaches 100%.
- Validation accuracy stops improving or decreases.

This behaviour is known as **overfitting** and indicates that the model is memorising the training data rather than learning general features.

---

### Underfitting

If both the training and validation accuracy remain low, the model may not be powerful enough to learn the task.

Possible solutions include:

- Training for more epochs.
- Increasing the model capacity.
- Improving the quality of the dataset.

## 🧪 8. Evaluating the Model on the Test Set

After training a machine learning model, we need an unbiased estimate of how well it performs on **new, unseen data**.

Throughout training, we used the **validation dataset** to monitor the model's performance and select the best-performing model. However, repeatedly evaluating on the validation set can unintentionally influence our model design and training decisions.

To obtain a fair assessment, we now evaluate the model on the **test dataset**, which has not been used during training or model selection.

Before evaluating, we load the checkpoint corresponding to the best validation accuracy that was saved during training.


In [ ]:
# We load the saved model from memory and use it to evaluate the performance on unseen test

best_model = TinyVGG(
    input_channels=3,
    hidden_units=32,
    num_classes=NUM_CLASSES,
).to(device)

best_model.load_state_dict(
    torch.load(
        CHECKPOINT_DIR / "tinyvgg_best_model.pth",
        map_location=device,
    )
)

best_model.eval()

In [ ]:
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score,
)


@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
):

    model.eval()

    accuracy = MulticlassAccuracy(
        num_classes=NUM_CLASSES
    ).to(device)

    precision = MulticlassPrecision(
        num_classes=NUM_CLASSES,
        average="macro",
    ).to(device)

    recall = MulticlassRecall(
        num_classes=NUM_CLASSES,
        average="macro",
    ).to(device)

    f1 = MulticlassF1Score(
        num_classes=NUM_CLASSES,
        average="macro",
    ).to(device)

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        accuracy.update(predictions, labels)
        precision.update(predictions, labels)
        recall.update(predictions, labels)
        f1.update(predictions, labels)

    return {
        "Accuracy": accuracy.compute().item(),
        "Precision": precision.compute().item(),
        "Recall": recall.compute().item(),
        "F1 Score": f1.compute().item(),
    }

In [ ]:
results = evaluate_model(
    best_model,
    test_loader,
    device,
)

for metric, value in results.items():
    print(f"{metric:12s}: {value:.3f}")

###  📊 8.1. Understanding the Evaluation Metrics

A single metric rarely tells the whole story.

For this reason, we evaluate the model using several complementary metrics.

| Metric | What it Measures |
|----------|-----------------|
| **Accuracy** | The proportion of correctly classified images. |
| **Precision** | How many predicted examples of a class are actually correct. |
| **Recall** | How many examples of a class the model successfully identifies. |
| **F1-score** | A balance between precision and recall. |

Together, these metrics provide a more complete picture of the model's performance than accuracy alone.

## 🖼️ 9. Visualising Model Predictions

Numerical metrics provide a useful summary of model performance, but they do not tell us **which images the model classified correctly or incorrectly**.

A powerful way to understand a model's behaviour is to examine its predictions directly.

In this section, we will display a selection of test images together with:

- The **ground truth** species.
- The **predicted** species.
- Whether the prediction was correct.

Visualising predictions helps us identify patterns in the model's successes and mistakes that may not be obvious from numerical evaluation alone.

In [ ]:
import matplotlib.pyplot as plt
import torch


@torch.inference_mode()
def plot_predictions(
    model,
    dataloader,
    class_names,
    device,
    num_images=32,
):
    """
    Plot a grid of images showing the ground truth and predicted labels.

    Parameters
    ----------
    model : nn.Module
    dataloader : DataLoader
        Usually the validation or test dataloader.
    class_names : list[str]
    device : torch.device
    num_images : int
    """

    model.eval()

    images_plotted = 0

    fig, axes = plt.subplots(4, 8, figsize=(16, 10))
    axes = axes.flatten()

    for images, labels in dataloader:

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu()

        images = images.cpu()

        for image, true_label, pred_label in zip(images, labels, predictions):

            if images_plotted >= num_images:
                plt.tight_layout()
                plt.show()
                return

            ax = axes[images_plotted]

            # Convert tensor to HWC format
            image = image.permute(1, 2, 0)

            ax.imshow(image)

            correct = true_label == pred_label

            ax.set_title(
                f"True: {class_names[true_label]}\n"
                f"Pred: {class_names[pred_label]}",
                fontsize=9,
                color="green" if correct else "red",
            )

            ax.axis("off")

            images_plotted += 1

    plt.tight_layout()
    plt.show()

In [ ]:
plot_predictions(
    model=model,
    dataloader=test_loader,
    class_names=test_dataset.classes,
    device=device,
)

###  🔭 9.1. Looking Beyond the Numbers

As you examine the predictions, ask yourself the following questions:

- Are the incorrect predictions understandable?
- Do some animal species have similar visual features?
- Are some animals partially hidden by vegetation?
- Is the image blurry or poorly illuminated?
- Is the animal very small or far from the camera?

Machine learning models do not simply make random mistakes. In many cases, incorrect predictions reveal genuine challenges within the dataset.

Understanding these challenges is an important part of developing reliable computer vision systems.

# 🎓 Summary

Congratulations! In this practical, you have completed the full workflow for training an image classification model using PyTorch.

Starting from a collection of camera trap images, you:

- Explored and understood the dataset.
- Applied image preprocessing techniques.
- Created training, validation and test datasets.
- Built a Convolutional Neural Network from scratch.
- Trained the model using gradient descent.
- Evaluated its performance.
- Visualised its predictions on unseen images.

Although the TinyVGG model is relatively simple, it demonstrates the fundamental workflow used by many modern computer vision systems.

The same principles apply to larger architectures such as ResNet, EfficientNet and Vision Transformers, which you will encounter later on.


**THANK YOU FOR YOUR TIME**